# Hidden Evaluation Log — Stage 2

This notebook is deliberately limited to **loading, predicting, and evaluating**.

The model itself is the same serialized Stage 1 checkpoint:

```text
model_checkpoint/sentiment_ensemble.joblib
```

Nothing in this notebook fits a vectorizer, trains an SVM, changes the ensemble weight, or moves the decision threshold.


## Evaluation map

I organized Stage 2 as a short verification chain:

**checkpoint identity**  
→ **hidden-set inspection**  
→ **frozen inference**  
→ **performance ledger**  
→ **public/hidden comparison**  
→ **prediction export**

The goal is to make it obvious where evaluation begins and to keep every training operation out of Stage 2.


In [ ]:
from pathlib import Path
import hashlib
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

frozen_path = Path("model_checkpoint") / "sentiment_ensemble.joblib"

# Accept either filename used by the course release.
hidden_path = (
    Path("hidden_test_with_labels.csv")
    if Path("hidden_test_with_labels.csv").exists()
    else Path("hidden_test.csv")
)

assert frozen_path.exists(), "Stage 1 checkpoint not found."
assert hidden_path.exists(), "Hidden evaluation file not found."

print("checkpoint file:", frozen_path)
print("hidden file:", hidden_path)


# I. Identify the frozen checkpoint

Before making any prediction, I fingerprint the submitted model file.

This is not part of the classifier. It is simply a way to record which exact checkpoint was used for hidden evaluation.


In [ ]:
checkpoint_fingerprint = hashlib.sha256(
    frozen_path.read_bytes()
).hexdigest()

print("checkpoint SHA-256")
print(checkpoint_fingerprint)


# II. Inspect the hidden release

The hidden labels are available only for Stage 2 evaluation. They are read here so I can calculate accuracy and the confusion matrix after predictions are produced.


In [ ]:
hidden_bank = pd.read_csv(hidden_path)

print("rows:", len(hidden_bank))
print("columns:", list(hidden_bank.columns))
print()
print("label counts:")
print(hidden_bank["label"].value_counts().sort_index())


The hidden set is balanced, so ordinary accuracy and balanced accuracy should be directly comparable. That also makes the public-vs-hidden comparison easier to interpret.


# III. Load once, predict once

The Stage 1 bundle already contains:

- the fitted word TF-IDF pipeline,
- the fitted character TF-IDF pipeline,
- the word/character ensemble weight,
- the fixed decision threshold.

Stage 2 only calls `decision_function()` on the two saved classifiers and then applies the already-stored combination rule.


In [ ]:
sealed_stage1 = joblib.load(frozen_path)

hidden_text = hidden_bank["text"].fillna("").astype(str)
hidden_gold = hidden_bank["label"].astype(int).to_numpy()

word_margin = sealed_stage1["word_model"].decision_function(hidden_text)
char_margin = sealed_stage1["char_model"].decision_function(hidden_text)

combined_margin = (
    sealed_stage1["alpha"] * word_margin
    + (1.0 - sealed_stage1["alpha"]) * char_margin
)

hidden_prediction = (
    combined_margin >= sealed_stage1["threshold"]
).astype(int)

print("predictions generated:", len(hidden_prediction))
print("stored alpha:", sealed_stage1["alpha"])
print("stored threshold:", sealed_stage1["threshold"])


## Sanity checks

These checks are here to catch accidental output-format problems before evaluation or export.


In [ ]:
assert len(hidden_prediction) == len(hidden_bank)
assert set(np.unique(hidden_prediction)).issubset({0, 1})

print("prediction checks passed")


# IV. Performance ledger

I report both total accuracy and balanced accuracy, followed by a full classification report and confusion matrix.


In [ ]:
hidden_acc = accuracy_score(hidden_gold, hidden_prediction)
hidden_bal_acc = balanced_accuracy_score(hidden_gold, hidden_prediction)
hidden_cm = confusion_matrix(hidden_gold, hidden_prediction)

print(f"accuracy:          {hidden_acc:.4f}")
print(f"balanced accuracy: {hidden_bal_acc:.4f}")
print()
print(classification_report(
    hidden_gold,
    hidden_prediction,
    target_names=["negative", "positive"],
    digits=4,
))
print("confusion matrix:")
print(hidden_cm)


In [ ]:
ConfusionMatrixDisplay(
    confusion_matrix=hidden_cm,
    display_labels=["negative", "positive"],
).plot(values_format="d")

plt.title("Hidden Test Confusion Matrix")
plt.show()


## Hidden-set result

The frozen Stage 1 model reaches **77.0% accuracy** on the 600-review hidden test set.

```text
[[225, 75],
 [ 63,237]]
```

Interpreting the matrix by row:

- 225 negative reviews were classified correctly.
- 75 negative reviews were classified as positive.
- 63 positive reviews were classified as negative.
- 237 positive reviews were classified correctly.

Because the hidden set contains 300 examples from each class, the model is slightly better on positive reviews than on negative reviews, but the gap is not extreme.


# V. Public result vs. hidden result

The Stage 1 public-test accuracy was **77.0%**.

The Stage 2 hidden-test accuracy is also **77.0%**.

The exact match in total accuracy is useful because it suggests the public score was not just an unusually favorable result. The confusion matrices are not identical, so the model made a different mix of mistakes on the hidden data, but its overall level of performance remained stable.


# VI. What I would test next

With more time or compute, I would mainly test methods that improve representation quality without depending on a much larger labeled dataset.

The first comparison I would make is against a compact pretrained language model or pretrained sentence embeddings. I would still keep model selection restricted to the original training data.

I would also try repeated stratified cross-validation instead of relying on one 5-fold split. With only 240 labeled reviews, repeating the folds could give a more stable estimate of which settings generalize best.


# VII. Export the required hidden predictions

The submission file must contain exactly:

```text
id,predicted_label
```


In [ ]:
hidden_submission = pd.DataFrame({
    "id": hidden_bank["id"],
    "predicted_label": hidden_prediction.astype(int),
})

assert list(hidden_submission.columns) == ["id", "predicted_label"]
assert hidden_submission["predicted_label"].isin([0, 1]).all()
assert len(hidden_submission) == len(hidden_bank)

hidden_submission.to_csv(
    "hidden_test_predictions.csv",
    index=False,
)

hidden_submission.head()


# Use of AI

Generative AI was used as a support tool during the implementation and presentation of this evaluation.

The AI assistance was focused on **evaluation workflow and documentation**, not on changing the trained model. The main areas of assistance were:

- organizing the inference code so the saved word model, character model, ensemble weight, and threshold were reused exactly as stored;
- verifying that the hidden prediction file followed the required two-column format;
- helping calculate and present total accuracy, balanced accuracy, the classification report, and the confusion matrix;
- helping compare the public-test result with the hidden-test result in a concise way;
- helping add a checkpoint fingerprint so the evaluated model file could be identified explicitly;
- helping reorganize and rewrite the notebook so the evaluation workflow is easier to audit.

Examples of the kinds of requests made to the AI were about how to structure inference-only code, how to verify the prediction CSV, how to present the evaluation metrics clearly, and how to explain the relationship between public and hidden performance.

AI was **not** used to relabel hidden examples, rewrite review text, choose hidden-set labels manually, retrain the classifiers, fine-tune the model, change the stored ensemble weight, or adjust the decision threshold after seeing hidden performance.

The hidden labels are used only after predictions are generated so that evaluation metrics can be calculated.
